# TS2Vec V2 for UCR classification

This notebook keeps the official TS2Vec self-supervised encoder and adapts it to the
unified Two-Tower V2 classification protocol.

- Pretraining uses only the official TRAIN subset.
- The linear probe uses one-hot labels and BCEWithLogitsLoss.
- The best probe checkpoint is selected by validation BCE.
- The official TEST set is evaluated once after restoring that checkpoint.
- Aout files are marker files only for a matched cohort; Aout values are never used.

In [ ]:
from __future__ import annotations
import copy, json, os, platform, random, shutil, sys, tempfile, traceback, urllib.request, zipfile
from datetime import datetime, timezone
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.backends.cudnn as cudnn
import torch.nn as nn
import torch.optim as optim
from sklearn.metrics import accuracy_score, roc_auc_score
from sklearn.preprocessing import label_binarize
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.utils.data import Dataset, DataLoader

DATA_ROOT = Path(r"D:\2025暑期科研\UCRArchive_2018\UCRArchive_2018")
OUTPUT_ROOT = Path(r"D:\2025暑期科研\UCRArchive_2018\TwoTower_Project_V2\final_runs_v2")

MODEL_NAME = "ts2vec"
EXPERIMENT_VERSION = "ts2vec_v2_official_ssl_linear_probe_bce_correct_metrics"
SEED = 42
BATCH_SIZE = 8
NUM_EPOCHS = 60
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 0.0
T_MAX = 50
VAL_FRACTION = 0.20
MIN_EFFECTIVE_LENGTH = 8
NUM_WORKERS = 0

TS2VEC_OUTPUT_DIMS = 256  # alternatives: 320 (official default), 128, 64
TS2VEC_HIDDEN_DIMS = 32   # alternatives: 64 (official default), 48, 16
TS2VEC_DEPTH = 6          # alternatives: 10 (official default), 8, 4
TS2VEC_TEMPORAL_UNIT = 0

REQUIRE_AOUT_MARKERS = True
RUN_SMOKE_TEST = False
RUN_FULL_EXPERIMENT = True
SMOKE_DATASETS = ["Coffee", "ArrowHead"]
SMOKE_EPOCHS = 3
RESUME_COMPLETED_DATASETS = True
SAVE_INDIVIDUAL_CURVES = True
DEVICE_REQUEST = "cuda:0"

# Official TS2Vec source is kept in a fixed, easy-to-find project folder.
TS2VEC_REPO_DIR = OUTPUT_ROOT.parent / "external" / "ts2vec_official"
TS2VEC_GITHUB_ZIP = "https://github.com/zhihanyue/ts2vec/archive/refs/heads/main.zip"

def ensure_official_ts2vec_source():
    """Download the official repository once if ts2vec.py is not present."""
    if (TS2VEC_REPO_DIR / "ts2vec.py").exists():
        return TS2VEC_REPO_DIR
    TS2VEC_REPO_DIR.parent.mkdir(parents=True, exist_ok=True)
    with tempfile.TemporaryDirectory() as td:
        archive = Path(td) / "ts2vec-main.zip"
        print("Downloading official TS2Vec source to:", TS2VEC_REPO_DIR)
        urllib.request.urlretrieve(TS2VEC_GITHUB_ZIP, archive)
        with zipfile.ZipFile(archive) as zf:
            zf.extractall(td)
        extracted = Path(td) / "ts2vec-main"
        if not (extracted / "ts2vec.py").exists():
            raise FileNotFoundError("Official TS2Vec archive has unexpected structure.")
        shutil.copytree(extracted, TS2VEC_REPO_DIR)
    return TS2VEC_REPO_DIR
ensure_official_ts2vec_source()
if str(TS2VEC_REPO_DIR) not in sys.path:
    sys.path.insert(0, str(TS2VEC_REPO_DIR))

try:
    from ts2vec import TS2Vec
except ImportError as exc:
    raise ImportError(
        "TS2Vec source was not found. Set TS2VEC_REPO_DIR to the folder containing "
        "official ts2vec.py plus models/ and utils.py."
    ) from exc

def set_global_seed(seed=SEED):
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    cudnn.deterministic = True; cudnn.benchmark = False

def resolve_device(requested):
    return torch.device(requested) if requested.startswith("cuda") and torch.cuda.is_available() else torch.device("cpu")

set_global_seed()
DEVICE = resolve_device(DEVICE_REQUEST)
print("Python:", platform.python_version())
print("PyTorch:", torch.__version__)
print("Resolved device:", DEVICE)
if torch.cuda.is_available(): print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
def model_output_dir(debug=False):
    return OUTPUT_ROOT / ("_debug" if debug else "") / MODEL_NAME / f"seed_{SEED}"

def ensure_output_tree(base):
    paths = {"base":base, "checkpoints":base/"checkpoints", "curves":base/"curves",
             "dataset_histories":base/"dataset_histories",
             "shared_splits":OUTPUT_ROOT/"_shared_splits"/f"seed_{SEED}"}
    for p in paths.values(): p.mkdir(parents=True, exist_ok=True)
    return paths

def atomic_write_csv(df, path):
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix+".tmp"); df.to_csv(tmp,index=False,encoding="utf-8"); os.replace(tmp,path)

def atomic_write_json(obj, path):
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix+".tmp")
    with open(tmp,"w",encoding="utf-8") as f: json.dump(obj,f,ensure_ascii=False,indent=2)
    os.replace(tmp,path)

def atomic_torch_save(obj, path):
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix+".tmp"); torch.save(obj,tmp); os.replace(tmp,path)

def load_ckpt(path):
    try: return torch.load(path,map_location=DEVICE,weights_only=True)
    except TypeError: return torch.load(path,map_location=DEVICE)

def append_rows(path, new_df, dataset):
    old = pd.read_csv(path) if path.exists() else pd.DataFrame()
    if not old.empty and "dataset" in old.columns: old=old[old.dataset!=dataset]
    out = pd.concat([old,new_df],ignore_index=True)
    cols=[c for c in ["dataset","epoch"] if c in out.columns]
    if cols: out=out.sort_values(cols).reset_index(drop=True)
    atomic_write_csv(out,path); return out

def make_class_safe_split(labels, dataset, split_dir, val_fraction=VAL_FRACTION):
    split_dir.mkdir(parents=True,exist_ok=True)
    path=split_dir/f"{dataset}_split_seed_{SEED}.npz"
    if path.exists():
        z=np.load(path); return z["train_idx"],z["val_idx"],path
    rng=np.random.RandomState(SEED); tr=[]; va=[]
    for cls in np.unique(labels):
        idx=np.flatnonzero(labels==cls); rng.shuffle(idx)
        n=max(1,round(len(idx)*val_fraction)) if len(idx)>1 else 0
        n=min(n,len(idx)-1) if len(idx)>1 else 0
        va.extend(idx[:n]); tr.extend(idx[n:])
    tr=np.asarray(sorted(tr),dtype=np.int64); va=np.asarray(sorted(va),dtype=np.int64)
    if len(va)==0: raise ValueError("Validation split is empty.")
    np.savez(path,train_idx=tr,val_idx=va); return tr,va,path

def clean_and_pad(raw, min_len=MIN_EFFECTIVE_LENGTH, fixed_len=None):
    rows=[]; keep=[]; lengths=[]
    for i,row in enumerate(raw):
        v=row[~np.isnan(row)]
        if len(v)<min_len: continue
        v=(v-float(v.mean()))/(float(v.std()) if float(v.std())>0 else 1.0)
        rows.append(v); keep.append(i); lengths.append(len(v))
    if not rows: raise ValueError("All samples were filtered out.")
    target=int(fixed_len if fixed_len is not None else max(lengths))
    out=[v[:target] if len(v)>=target else np.pad(v,(0,target-len(v))) for v in rows]
    return np.stack(out).astype("float32"),np.asarray(keep,dtype=np.int64)

class EmbeddingDataset(Dataset):
    def __init__(self,x,y): self.x=x.astype("float32"); self.y=y.astype("float32")
    def __len__(self): return len(self.y)
    def __getitem__(self,i): return self.x[i],self.y[i]

def corrected_metrics(logits, labels):
    logits=np.asarray(logits); labels=np.asarray(labels)
    if logits.ndim==1: logits=logits[:,None]
    if labels.ndim==1: labels=labels[:,None]
    probs=1/(1+np.exp(-np.clip(logits,-50,50)))
    if logits.shape[1]==1:
        yt=labels[:,0].astype(int); yp=(probs[:,0]>=.5).astype(int)
        auc=roc_auc_score(yt,probs[:,0]) if np.unique(yt).size==2 else float("nan")
        return float(auc),float(accuracy_score(yt,yp))
    yt=labels.argmax(1); yp=probs.argmax(1)
    aucs=[roc_auc_score(labels[:,j],probs[:,j]) for j in range(labels.shape[1]) if np.unique(labels[:,j]).size==2]
    return float(np.mean(aucs)) if aucs else float("nan"),float(accuracy_score(yt,yp))

def normalize_embeddings(z):
    z=np.asarray(z,dtype="float32")
    if z.ndim==3: z=z[:,0,:] if z.shape[1]==1 else z.max(axis=1)
    if z.ndim!=2: raise ValueError(f"Unexpected TS2Vec shape: {z.shape}")
    return z

def fit_encoder(train_series, epochs):
    model=TS2Vec(input_dims=1,output_dims=TS2VEC_OUTPUT_DIMS,hidden_dims=TS2VEC_HIDDEN_DIMS,
                 depth=TS2VEC_DEPTH,device=str(DEVICE),lr=LEARNING_RATE,batch_size=BATCH_SIZE,
                 max_train_length=None,temporal_unit=TS2VEC_TEMPORAL_UNIT)
    log=model.fit(train_data=train_series,n_iters=None,n_epochs=int(epochs),verbose=False)
    return model,log

def eval_loss(model, loader, criterion):
    model.eval(); total=n=0
    with torch.no_grad():
        for x,y in loader:
            x=x.to(DEVICE); y=y.to(DEVICE); total+=float(criterion(model(x),y).item())*len(y); n+=len(y)
    return total/max(1,n)

def eval_test_once(model, loader, criterion):
    model.eval(); logs=[]; labs=[]; total=n=0
    with torch.no_grad():
        for x,y in loader:
            x=x.to(DEVICE); yd=y.to(DEVICE); out=model(x)
            total+=float(criterion(out,yd).item())*len(y); n+=len(y)
            logs.append(out.cpu().numpy()); labs.append(y.numpy())
    auc,acc=corrected_metrics(np.concatenate(logs),np.concatenate(labs))
    return auc,acc,total/max(1,n),n

a,a_acc=corrected_metrics(np.array([[-2],[2],[1],[-1]]),np.array([[0],[1],[0],[1]]))
assert np.isclose(a_acc,.5)
assert normalize_embeddings(np.zeros((3,1,5))).shape==(3,5)
print(f"TS2Vec V2 self-checks passed | binary ACC={a_acc:.2f} | AUC={a:.2f}")

In [ ]:
def run_one_dataset_v2(dataset_dir, dataset_name, paths, epochs=NUM_EPOCHS, save_curve=SAVE_INDIVIDUAL_CURVES):
    set_global_seed()
    train_tsv=dataset_dir/f"{dataset_name}_TRAIN_cleaned.tsv"; test_tsv=dataset_dir/f"{dataset_name}_TEST_cleaned.tsv"
    required=[train_tsv,test_tsv]
    if REQUIRE_AOUT_MARKERS: required += [dataset_dir/f"{dataset_name}_Aout_train_k2.csv",dataset_dir/f"{dataset_name}_Aout_test_k2.csv"]
    missing=[str(p) for p in required if not p.exists()]
    if missing: raise FileNotFoundError("Missing required files: "+"; ".join(missing))
    tr=pd.read_csv(train_tsv,sep="	",header=None); te=pd.read_csv(test_tsv,sep="	",header=None)
    ytr_raw=tr.iloc[:,0].to_numpy(); yte_raw=te.iloc[:,0].to_numpy()
    xtr_raw=tr.iloc[:,1:].to_numpy(dtype="float32"); xte_raw=te.iloc[:,1:].to_numpy(dtype="float32")
    tmp,_=clean_and_pad(xtr_raw); seq_len=tmp.shape[1]
    xtr,ktr=clean_and_pad(xtr_raw,fixed_len=seq_len); xte,kte=clean_and_pad(xte_raw,fixed_len=seq_len)
    ytr=ytr_raw[ktr]; yte=yte_raw[kte]; classes=np.sort(np.unique(ytr))
    if len(classes)<2: raise ValueError("Training data has fewer than two classes.")
    unseen=np.setdiff1d(np.unique(yte),classes)
    if len(unseen): raise ValueError(f"TEST contains unseen labels: {unseen}")
    Ytr=label_binarize(ytr,classes=classes).astype("float32"); Yte=label_binarize(yte,classes=classes).astype("float32")
    if Ytr.ndim==1: Ytr=Ytr[:,None]; Yte=Yte[:,None]
    tri,vai,split_path=make_class_safe_split(ytr,dataset_name,paths["shared_splits"])
    train_series=xtr[tri][:,:,None].astype("float32"); val_series=xtr[vai][:,:,None].astype("float32"); test_series=xte[:,:,None].astype("float32")
    print(f"[{dataset_name}] TS2Vec pretraining on TRAIN subset only | train/val/test={len(tri)}/{len(vai)}/{len(test_series)}")
    encoder,prelog=fit_encoder(train_series,epochs)
    ztr=normalize_embeddings(encoder.encode(train_series,encoding_window="full_series"))
    zva=normalize_embeddings(encoder.encode(val_series,encoding_window="full_series"))
    zte=normalize_embeddings(encoder.encode(test_series,encoding_window="full_series"))
    train_ds=EmbeddingDataset(ztr,Ytr[tri]); val_ds=EmbeddingDataset(zva,Ytr[vai]); test_ds=EmbeddingDataset(zte,Yte)
    gen=torch.Generator(device="cpu").manual_seed(SEED); pin=DEVICE.type=="cuda"
    trl=DataLoader(train_ds,batch_size=BATCH_SIZE,shuffle=True,generator=gen,pin_memory=pin,num_workers=NUM_WORKERS)
    val=DataLoader(val_ds,batch_size=BATCH_SIZE,shuffle=False,pin_memory=pin,num_workers=NUM_WORKERS)
    tel=DataLoader(test_ds,batch_size=BATCH_SIZE,shuffle=False,pin_memory=pin,num_workers=NUM_WORKERS)
    head=nn.Linear(ztr.shape[1],Ytr.shape[1]).to(DEVICE); params=sum(p.numel() for p in head.parameters() if p.requires_grad)
    criterion=nn.BCEWithLogitsLoss(); opt=optim.Adam(head.parameters(),lr=LEARNING_RATE,weight_decay=WEIGHT_DECAY); sch=CosineAnnealingLR(opt,T_max=T_MAX,eta_min=0)
    ckpt=paths["checkpoints"]/f"{dataset_name}_best.pt"; best=float("inf"); best_epoch=0; hist=[]
    for ep in range(1,epochs+1):
        head.train(); total=n=0
        for x,y in trl:
            x=x.to(DEVICE); y=y.to(DEVICE); opt.zero_grad(set_to_none=True); loss=criterion(head(x),y); loss.backward(); opt.step()
            total+=float(loss.item())*len(y); n+=len(y)
        train_loss=total/max(1,n); val_loss=eval_loss(head,val,criterion)
        hist.append({"dataset":dataset_name,"model":MODEL_NAME,"seed":SEED,"epoch":ep,"train_loss":train_loss,"val_loss":val_loss,"learning_rate":opt.param_groups[0]["lr"]})
        if val_loss<best:
            best=float(val_loss); best_epoch=ep
            atomic_torch_save({"state_dict":copy.deepcopy(head.state_dict()),"best_epoch":ep,"best_val_loss":best,
                               "classes":classes.tolist(),"embedding_dim":int(ztr.shape[1]),"parameter_count":params,
                               "sequence_length":int(seq_len),"split_path":str(split_path),
                               "ts2vec_output_dims":TS2VEC_OUTPUT_DIMS,"ts2vec_hidden_dims":TS2VEC_HIDDEN_DIMS,"ts2vec_depth":TS2VEC_DEPTH},ckpt)
        sch.step()
    history=pd.DataFrame(hist); atomic_write_csv(history,paths["dataset_histories"]/f"{dataset_name}.csv")
    head.load_state_dict(load_ckpt(ckpt)["state_dict"]); set_global_seed()
    auc,acc,loss,n_test=eval_test_once(head,tel,criterion)
    if save_curve:
        fig,ax=plt.subplots(figsize=(6.2,4)); ax.plot(history.epoch,history.train_loss,label="Probe Train BCE"); ax.plot(history.epoch,history.val_loss,label="Probe Validation BCE"); ax.axvline(best_epoch,color="black",ls="--",lw=1,label="Best epoch"); ax.set(title=f"{dataset_name} - TS2Vec V2",xlabel="Probe epoch",ylabel="BCE loss"); ax.grid(alpha=.25); ax.legend(); fig.tight_layout(); fig.savefig(paths["curves"]/f"{dataset_name}_loss.png",dpi=180); plt.close(fig)
    pre_last=float(prelog[-1]) if prelog is not None and len(prelog) else float("nan")
    result={"dataset":dataset_name,"model":MODEL_NAME,"seed":SEED,"test_auc":auc,"test_acc":acc,"test_loss":loss,"n_samples":n_test,
            "num_classes":int(len(classes)),"num_outputs":int(Ytr.shape[1]),"best_epoch":int(best_epoch),"best_val_loss":float(best),
            "train_samples":int(len(train_ds)),"val_samples":int(len(val_ds)),"parameter_count":int(params),
            "ts2vec_pretrain_epochs":int(epochs),"ts2vec_pretrain_last_loss":pre_last,"uses_aout":False,"status":"completed"}
    print(f"[{dataset_name}] TEST loss={loss:.4f} | AUC={auc:.4f} | ACC={acc:.4f} | best_probe_epoch={best_epoch} | n={n_test}")
    return result,history

def discover_datasets(root):
    if not root.exists(): raise FileNotFoundError(f"DATA_ROOT does not exist: {root}")
    names=[]
    for d in sorted(p for p in root.iterdir() if p.is_dir()):
        name=d.name; req=[d/f"{name}_TRAIN_cleaned.tsv",d/f"{name}_TEST_cleaned.tsv"]
        if REQUIRE_AOUT_MARKERS: req += [d/f"{name}_Aout_train_k2.csv",d/f"{name}_Aout_test_k2.csv"]
        if all(p.exists() for p in req): names.append(name)
    return names

def summarize(df):
    valid=df.dropna(subset=["test_auc"]); total=float(df.n_samples.sum()); auc_n=float(valid.n_samples.sum())
    return {"datasets_completed":int(len(df)),"simple_mean_auc":float(valid.test_auc.mean()),
            "simple_mean_acc":float(df.test_acc.mean()),"simple_mean_loss":float(df.test_loss.mean()),
            "weighted_auc":float((valid.test_auc*valid.n_samples).sum()/auc_n),
            "weighted_acc":float((df.test_acc*df.n_samples).sum()/total)}

def run_datasets_v2(root,selected=None,debug=False,epochs=NUM_EPOCHS,resume=RESUME_COMPLETED_DATASETS):
    base=model_output_dir(debug); paths=ensure_output_tree(base); final=base/"final_results.csv"; histories=base/"histories.csv"; failed=base/"failed_datasets.csv"
    atomic_write_json({"experiment_version":EXPERIMENT_VERSION,"model":MODEL_NAME,"seed":SEED,"debug":debug,"data_root":str(root),
                       "output_root":str(OUTPUT_ROOT),"uses_aout":False,"aout_markers_only":REQUIRE_AOUT_MARKERS,
                       "epochs":epochs,"batch_size":BATCH_SIZE,"learning_rate":LEARNING_RATE,"val_fraction":VAL_FRACTION,
                       "loss":"BCEWithLogitsLoss","checkpoint_selection":"minimum validation BCE of linear probe",
                       "test_evaluation":"once after restoring best checkpoint","ts2vec_output_dims":TS2VEC_OUTPUT_DIMS,
                       "ts2vec_hidden_dims":TS2VEC_HIDDEN_DIMS,"ts2vec_depth":TS2VEC_DEPTH,
                       "created_utc":datetime.now(timezone.utc).isoformat()},base/"run_config.json")
    available=discover_datasets(root); names=available if selected is None else [x for x in selected if x in available]
    if selected is not None:
        missing=sorted(set(selected)-set(available))
        if missing: raise FileNotFoundError(f"Selected datasets not discoverable: {missing}")
    completed=set()
    if resume and final.exists():
        old=pd.read_csv(final)
        if "dataset" in old.columns and "status" in old.columns: completed=set(old.loc[old.status=="completed","dataset"])
    print(f"Output directory: {base}"); print(f"Datasets requested: {len(names)}"); print(f"Already completed and skipped: {len(completed.intersection(names))}")
    for pos,name in enumerate(names,1):
        if name in completed: print(f"[{pos}/{len(names)}] Skip completed: {name}"); continue
        print(f"\n[{pos}/{len(names)}] Start: {name}")
        try:
            result,history=run_one_dataset_v2(root/name,name,paths,epochs,SAVE_INDIVIDUAL_CURVES)
            append_rows(final,pd.DataFrame([result]),name); append_rows(histories,history,name)
            if failed.exists(): atomic_write_csv(pd.read_csv(failed).query("dataset != @name"),failed)
        except Exception as error:
            failure=pd.DataFrame([{"dataset":name,"model":MODEL_NAME,"seed":SEED,"error_type":type(error).__name__,"error_message":str(error),"traceback":traceback.format_exc(),"recorded_utc":datetime.now(timezone.utc).isoformat()}])
            append_rows(failed,failure,name); print(f"FAILED {name}: {type(error).__name__}: {error}")
    if not final.exists(): raise RuntimeError("No dataset completed successfully.")
    out=pd.read_csv(final); out=out[out.dataset.isin(names)].sort_values("dataset").reset_index(drop=True); summary=summarize(out)
    summary.update({"model":MODEL_NAME,"seed":SEED,"debug":debug,"datasets_requested":len(names),"generated_utc":datetime.now(timezone.utc).isoformat()}); atomic_write_json(summary,base/"summary.json")
    print("\n========== TS2Vec V2 SUMMARY (OFFICIAL TEST, EVALUATED ONCE) =========="); print(out[["dataset","test_auc","test_acc","test_loss","n_samples"]].to_string(index=False))
    print(f"\nSimple mean: AUC={summary['simple_mean_auc']:.4f}, ACC={summary['simple_mean_acc']:.4f}, LOSS={summary['simple_mean_loss']:.4f}")
    print(f"Weighted: AUC={summary['weighted_auc']:.4f}, ACC={summary['weighted_acc']:.4f}")
    return out,summary

## Run order

1. First run with RUN_SMOKE_TEST=True and RUN_FULL_EXPERIMENT=False.
2. Confirm Coffee and ArrowHead finish and uses_aout=False is printed.
3. Restart the kernel, set RUN_SMOKE_TEST=False and RUN_FULL_EXPERIMENT=True, and run all cells again.

Smoke output is isolated in final_runs_v2/_debug/ts2vec/seed_42.
Full output is written to final_runs_v2/ts2vec/seed_42.

In [ ]:
if RUN_SMOKE_TEST:
    smoke_results, smoke_summary = run_datasets_v2(DATA_ROOT,selected=SMOKE_DATASETS,debug=True,epochs=SMOKE_EPOCHS,resume=False)
else:
    print("Smoke test disabled.")
if RUN_FULL_EXPERIMENT:
    final_results, final_summary = run_datasets_v2(DATA_ROOT,selected=None,debug=False,epochs=NUM_EPOCHS,resume=RESUME_COMPLETED_DATASETS)
else:
    print("Full experiment disabled.")

In [ ]:
# ===== Re-run five PermissionError datasets =====
import os
import time
import uuid
import copy
import torch

# 修复 Windows 下 os.replace() 偶发 PermissionError
def atomic_torch_save(obj, path, retries=8, delay=1.5):
    path.parent.mkdir(parents=True, exist_ok=True)

    # 每次使用独立临时文件，避免旧 .tmp 文件被占用
    tmp = path.with_name(
        f"{path.name}.{os.getpid()}.{uuid.uuid4().hex}.tmp"
    )

    try:
        for attempt in range(retries):
            try:
                torch.save(obj, tmp)
                os.replace(tmp, path)
                return
            except PermissionError as e:
                if attempt == retries - 1:
                    raise
                print(
                    f"[Checkpoint warning] PermissionError, "
                    f"retry {attempt + 1}/{retries}..."
                )
                time.sleep(delay * (attempt + 1))
    finally:
        if tmp.exists():
            try:
                tmp.unlink()
            except Exception:
                pass


# 之前因 checkpoint 保存失败的五个数据集
DATASETS_TO_RERUN = [
    "CricketZ",
    "HandOutlines",
    "Lightning7",
    "MixedShapesSmallTrain",
    "RefrigerationDevices",
]

print("Datasets to rerun:")
print(DATASETS_TO_RERUN)

rerun_results, rerun_summary = run_datasets_v2(
    root=DATA_ROOT,
    selected=DATASETS_TO_RERUN,
    debug=False,
    epochs=NUM_EPOCHS,
    resume=True,
)

print("\nFive-dataset rerun finished.")

In [ ]:
# ===== Recompute aggregate metrics for all completed datasets =====
from datetime import datetime, timezone

RESULT_DIR = OUTPUT_ROOT / MODEL_NAME / f"seed_{SEED}"
FINAL_RESULTS_PATH = RESULT_DIR / "final_results.csv"

all_results = pd.read_csv(FINAL_RESULTS_PATH)

# 当前实验中实际请求的全部数据集
requested_datasets = discover_datasets(DATA_ROOT)

# 只保留当前 125 个请求数据集，并且只统计 completed 结果
all_results = all_results[
    all_results["dataset"].isin(requested_datasets)
].copy()

if "status" in all_results.columns:
    all_results = all_results[
        all_results["status"] == "completed"
    ].copy()

all_results = all_results.drop_duplicates(
    subset=["dataset"],
    keep="last"
).sort_values("dataset").reset_index(drop=True)

# 去除缺失指标
valid = all_results.dropna(
    subset=["test_auc", "test_acc", "test_loss", "n_samples"]
).copy()

# Simple mean：每个数据集等权重
simple_mean_auc = float(valid["test_auc"].mean())
simple_mean_acc = float(valid["test_acc"].mean())
simple_mean_loss = float(valid["test_loss"].mean())

# Weighted mean：按照测试样本数量加权
weighted_auc = float(
    (valid["test_auc"] * valid["n_samples"]).sum()
    / valid["n_samples"].sum()
)

weighted_acc = float(
    (valid["test_acc"] * valid["n_samples"]).sum()
    / valid["n_samples"].sum()
)

completed_names = set(valid["dataset"])
failed_names = sorted(set(requested_datasets) - completed_names)

full_summary = {
    "model": MODEL_NAME,
    "seed": int(SEED),
    "datasets_requested": int(len(requested_datasets)),
    "datasets_completed": int(len(valid)),
    "datasets_failed": int(len(failed_names)),
    "failed_datasets": failed_names,
    "simple_mean_auc": simple_mean_auc,
    "simple_mean_acc": simple_mean_acc,
    "simple_mean_loss": simple_mean_loss,
    "weighted_auc": weighted_auc,
    "weighted_acc": weighted_acc,
    "generated_utc": datetime.now(timezone.utc).isoformat(),
}

# 覆盖为真正的全量汇总，而不是前五个数据集的局部汇总
atomic_write_json(
    full_summary,
    RESULT_DIR / "summary.json"
)

print("\n========== FINAL TS2Vec V2 SUMMARY ==========")
print(f"Datasets requested : {len(requested_datasets)}")
print(f"Datasets completed : {len(valid)}")
print(f"Datasets failed    : {len(failed_names)}")

print(
    f"\nSimple mean: "
    f"AUC={simple_mean_auc:.4f}, "
    f"ACC={simple_mean_acc:.4f}, "
    f"LOSS={simple_mean_loss:.4f}"
)

print(
    f"Weighted: "
    f"AUC={weighted_auc:.4f}, "
    f"ACC={weighted_acc:.4f}"
)

print("\nRemaining failed datasets:")
print(failed_names)